In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.float16,
    device_map="auto"
    )

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [23]:
def ask_assistant(question, system_prompt="you are a helpful, concise and funny Q&A assistant.", max_new_tokens=100, temperature=0.7, do_sample=True):
  messages=[
      {
          "role": "system",
          "content": system_prompt
      },
      {
          "role": "user",
          "content": question
      }
  ]

  inputs = tokenizer.apply_chat_template(
      messages,
      tokenizer=True,
      add_generation_prompt=True,
      return_tensors="pt"
  ).to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens= max_new_tokens,
      temperature= temperature,
      do_sample= do_sample
  )

  answer = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[-1]:],
      skip_special_tokens=True
  )

  return answer

In [27]:
ask_assistant("How are you doing?")

"I'm doing great! How about you?"

In [19]:
ask_assistant("why shrimps turn orange when boiled?")

"Shrimp turns orange when boiled because the water contains tannins, which give them their characteristic orange color. The tannins are responsible for the bright red color of the shrimp, not the boiling process itself. Boiling doesn't affect the tannin content in the water, so it's just an external factor causing the orange coloration."

In [28]:
ask_assistant("tell me a japanese joke")

'Why did the tomato turn red? Because it saw the salad dressing!'